In [1]:
import os


In [2]:
os.chdir("../")

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir:Path
    trained_model_path:Path
    updated_base_model_path:Path
    training_data:Path
    params_epochs:int
    params_batch_size:int
    params_is_augmentation: bool
    params_image_size:list

In [4]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories
import tensorflow as tf



[{'name': 'tensorflow', 'msg': 'From %s: The name %s is deprecated. Please use %s instead.\n', 'args': ('c:\\kidney disease classificaton\\myvenv\\Lib\\site-packages\\keras\\src\\losses.py:2976', 'tf.losses.sparse_softmax_cross_entropy', 'tf.compat.v1.losses.sparse_softmax_cross_entropy'), 'levelname': 'WARNING', 'levelno': 30, 'pathname': 'c:\\kidney disease classificaton\\myvenv\\Lib\\site-packages\\tensorflow\\python\\util\\module_wrapper.py', 'filename': 'module_wrapper.py', 'module': 'module_wrapper', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 149, 'funcName': '_tfmw_add_deprecation_warning', 'created': 1779197713.858547, 'msecs': 858.0, 'relativeCreated': 64730.23009300232, 'thread': 160072, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 161800, 'message': 'From c:\\kidney disease classificaton\\myvenv\\Lib\\site-packages\\keras\\src\\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_training_config(self) ->TrainingConfig:
        training=self.config.training
        prepare_base_model= self.config.prepare_base_model
        params=self.params
        training_data=os.path.join(self.config.data_ingestion.unzip_dir,"data")
        create_directories([
            Path(training.root_dir)])
        
        
        training_config=TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTAION,
            params_image_size=params.IMAGE_SIZE
        )
        
        return training_config    
        


In [6]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time


In [7]:
from tensorflow.keras import mixed_precision

# Now these lines will work perfectly
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

[{'name': 'tensorflow', 'msg': 'Mixed precision compatibility check (mixed_float16): WARNING\nThe dtype policy mixed_float16 may run slowly because this machine does not have a GPU. Only Nvidia GPUs with compute capability of at least 7.0 run quickly with mixed_float16.\nIf you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once', 'args': (), 'levelname': 'WARNING', 'levelno': 30, 'pathname': 'c:\\kidney disease classificaton\\myvenv\\Lib\\site-packages\\keras\\src\\mixed_precision\\device_compatibility_check.py', 'filename': 'device_compatibility_check.py', 'module': 'device_compatibility_check', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 121, 'funcName': '_log_device_compatibility_check', 'created': 1779197719.2109063, 'msecs': 210.0, 'relativeCreated': 70082.58938789368, 'thread': 160072, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process'

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from pathlib import Path
import tensorflow as tf
import numpy as np

class Training:
    def __init__(self, config):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
    def get_base_model(self):
        """Load .h5 TensorFlow model"""
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)
        print(f"✓ Loaded model: {self.config.updated_base_model_path}")
        
    def train_validator_generator(self):
        if self.config.params_is_augmentation:
            train_transforms = transforms.Compose([
                transforms.Resize(self.config.params_image_size[:-1]),
                transforms.RandomRotation(40),
                transforms.RandomHorizontalFlip(),
                transforms.RandomAffine(degrees=0, translate=(0.2, 0.2)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        else:
            train_transforms = transforms.Compose([
                transforms.Resize(self.config.params_image_size[:-1]),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        
        valid_transforms = transforms.Compose([
            transforms.Resize(self.config.params_image_size[:-1]),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
        
        full_dataset = datasets.ImageFolder(
            self.config.training_data, 
            transform=train_transforms
        )
        
        train_size = int(0.8 * len(full_dataset))
        valid_size = len(full_dataset) - train_size
        self.train_dataset, self.valid_dataset = random_split(
            full_dataset, 
            [train_size, valid_size],
            generator=torch.Generator().manual_seed(42)
        )
        
        self.valid_dataset.dataset.transform = valid_transforms
        
        self.train_loader = DataLoader(
            self.train_dataset,
            batch_size=self.config.params_batch_size,
            shuffle=True,
            num_workers=0,  # Set to 0 on Windows
            pin_memory=True
        )
        
        self.valid_loader = DataLoader(
            self.valid_dataset,
            batch_size=self.config.params_batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )
    
    @staticmethod
    def save_model(path: Path, model):
        model.save(path)
    
    def train(self):
        """Train using TensorFlow model with PyTorch DataLoader"""
        
        self.model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Helper to convert PyTorch tensors to TensorFlow format (B, H, W, C)
        def pytorch_to_tf_dataset(data_loader):
            for batch_x, batch_y in data_loader:
                # Convert to numpy and transpose (B, C, H, W) -> (B, H, W, C)
                batch_x = batch_x.numpy()
                batch_x = np.transpose(batch_x, (0, 2, 3, 1))
                
                # One-hot encode labels
                num_classes = self.model.layers[-1].output_shape[-1]
                batch_y = np.eye(num_classes)[batch_y.numpy()]
                
                yield batch_x, batch_y
        
        # FIXED: Using None for batch dimension to handle remainder batches
        output_sig = (
            tf.TensorSpec(
                shape=(None, 
                       self.config.params_image_size[0], 
                       self.config.params_image_size[1], 
                       self.config.params_image_size[2]), 
                dtype=tf.float32
            ),
            tf.TensorSpec(
                shape=(None, 
                       self.model.layers[-1].output_shape[-1]), 
                dtype=tf.float32
            )
        )

        train_tf_dataset = tf.data.Dataset.from_generator(
            lambda: pytorch_to_tf_dataset(self.train_loader),
            output_signature=output_sig
        ).cache().prefetch(buffer_size=tf.data.AUTOTUNE) # <--- ADD THIS LINE
        
        valid_tf_dataset = tf.data.Dataset.from_generator(
            lambda: pytorch_to_tf_dataset(self.valid_loader),
            output_signature=output_sig
        ).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
        valid_tf_dataset = tf.data.Dataset.from_generator(
            lambda: pytorch_to_tf_dataset(self.valid_loader),
            output_signature=output_sig
        )
        
        self.model.fit(
            train_tf_dataset,
            epochs=self.config.params_epochs,
            validation_data=valid_tf_dataset,
            verbose=1
        )
        
    
        self.save_model(self.config.trained_model_path, self.model)
        print(f"✓ Model saved to {self.config.trained_model_path}")

In [30]:
try:
    config=ConfigurationManager()
    training_config=config.get_training_config()
    training=Training(config=training_config)
    training.get_base_model()
    training.train_validator_generator()
    training.train()
except Exception as e:
    raise e

[{'name': 'cnnClassifierLogger', 'msg': 'yaml file:config\\config.yaml loaded successfully', 'args': (), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'c:\\kidney disease classificaton\\src\\cnnClassifier\\utils\\common.py', 'filename': 'common.py', 'module': 'common', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 30, 'funcName': 'read_yaml', 'created': 1779206848.4895978, 'msecs': 489.0, 'relativeCreated': 9199361.280918121, 'thread': 160072, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 161800, 'message': 'yaml file:config\\config.yaml loaded successfully'}(asctime)s:INFO:common:yaml file:config\config.yaml loaded successfully]
[{'name': 'cnnClassifierLogger', 'msg': 'yaml file:params.yaml loaded successfully', 'args': (), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'c:\\kidney disease classificaton\\src\\cnnClassifier\\utils\\common.py', 'filename': 'common.py', 'module': 'common', 'exc_info': None, 'exc_text': None, 'stack_info': Non

Using device: cuda
[{'name': 'tensorflow', 'msg': 'Error in loading the saved optimizer state. As a result, your model is starting with a freshly initialized optimizer.', 'args': (), 'levelname': 'WARNING', 'levelno': 30, 'pathname': 'c:\\kidney disease classificaton\\myvenv\\Lib\\site-packages\\keras\\src\\saving\\legacy\\hdf5_format.py', 'filename': 'hdf5_format.py', 'module': 'hdf5_format', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 255, 'funcName': 'load_model_from_hdf5', 'created': 1779206849.382549, 'msecs': 382.0, 'relativeCreated': 9200254.232168198, 'thread': 160072, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 161800, 'message': 'Error in loading the saved optimizer state. As a result, your model is starting with a freshly initialized optimizer.'}(asctime)s:WARNING:hdf5_format:Error in loading the saved optimizer state. As a result, your model is starting with a freshly initialized optimizer.]
✓ Loaded model: artifacts\prepare_ba

KeyboardInterrupt: 